### Building a RAG System with LangChain and ChromaDB
Retrieval-augmented generation (RAG) combines document retrieval with a language model. This example builds a small RAG workflow with:

- LangChain for loading, splitting, and chaining steps
- ChromaDB for local vector storage and retrieval
- OpenAI-compatible embeddings and chat models


In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

In [ ]:
## langchain imports
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_core.documents import Document
from langchain_core.embeddings import Embeddings
from sklearn.feature_extraction.text import HashingVectorizer

## vectorstores
from langchain_community.vectorstores import Chroma

## utility imports
import numpy as np
from typing import List

class HashEmbeddings(Embeddings):
    """Deterministic local fallback embeddings for offline notebook runs."""

    def __init__(self, n_features=1536):
        self.vectorizer = HashingVectorizer(
            n_features=n_features,
            alternate_sign=False,
            norm="l2",
            ngram_range=(1, 2),
        )

    def _embed(self, texts):
        return self.vectorizer.transform(texts).toarray().astype(float).tolist()

    def embed_documents(self, texts):
        return self._embed(texts)

    def embed_query(self, text):
        return self._embed([text])[0]

In [ ]:
# RAG Architecture Overview
print("""
RAG (Retrieval-Augmented Generation) Architecture:

1. Document Loading: Load documents from various sources
2. Document Splitting: Break documents into smaller chunks
3. Embedding Generation: Convert chunks into vector representations
4. Vector Storage: Store embeddings in ChromaDB
5. Query Processing: Convert user query to embedding
6. Similarity Search: Find relevant chunks from vector store
7. Context Augmentation: Combine retrieved chunks with query
8. Response Generation: LLM generates answer using context

Benefits of RAG:
- Reduces hallucinations
- Provides up-to-date information
- Allows citing sources
- Works with domain-specific knowledge
""")

### 1. Sample Data

In [ ]:
# Create sample documents
sample_docs = [
    """
    Machine Learning Fundamentals
    
    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from data without explicit rules for every case. There are three main 
    types of machine learning: supervised learning, unsupervised learning, and reinforcement 
    learning. Supervised learning uses labeled data to train models, while unsupervised 
    learning finds patterns in unlabeled data. Reinforcement learning learns through 
    interaction with an environment using rewards and penalties.
    """,
    
    """
    Deep Learning and Neural Networks
    
    Deep learning is a subset of machine learning based on artificial neural networks. 
    These networks are inspired by the human brain and consist of layers of interconnected 
    nodes. Deep learning is used in fields such as computer vision, natural language 
    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly 
    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers 
    excel at sequential data processing.
    """,
    
    """
    Natural Language Processing (NLP)
    
    NLP is a field of AI that focuses on the interaction between computers and human language. 
    Key tasks in NLP include text classification, named entity recognition, sentiment analysis, 
    machine translation, and question answering. Modern NLP heavily relies on transformer 
    architectures like BERT, GPT, and T5. These models use attention mechanisms to understand 
    context and relationships between words in text.
    """
]

sample_docs


In [ ]:
# Save sample documents to files
import tempfile
temp_dir=tempfile.mkdtemp()

for i,doc in enumerate(sample_docs):
    with open(f"{temp_dir}/doc_{i}.txt","w") as f:
        f.write(doc)

print(f"Sample document create in : {temp_dir}")

In [ ]:
print(f"Using sample documents from: {temp_dir}")

In [ ]:
temp_dir

### 2. Document Loading

In [ ]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

# Load documents from the temp directory created above.
loader = DirectoryLoader(
    temp_dir,
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
)
documents = loader.load()

print(f"Loaded {len(documents)} documents")
print()
print("First document preview:")
print(documents[0].page_content[:200] + "...")

In [ ]:
documents

### Document Splitting

In [ ]:
# Initialize text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,  # Maximum size of each chunk
    chunk_overlap=50,  # Overlap between chunks to maintain context
    length_function=len,
    separators=[" "]  # Hierarchy of separators
)
chunks=text_splitter.split_documents(documents)

print(f"Created {len(chunks)} chunks from {len(documents)} documents")
print(f"\nChunk example:")
print(f"Content: {chunks[0].page_content[:150]}...")
print(f"Metadata: {chunks[0].metadata}")

In [ ]:
chunks

### Embedding Models

In [ ]:
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if OPENAI_API_KEY:
    from langchain_openai import OpenAIEmbeddings
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
    print("Using OpenAIEmbeddings from the API.")
else:
    embeddings = HashEmbeddings(n_features=1536)
    print("OPENAI_API_KEY not found. Using deterministic local HashEmbeddings so the notebook still runs.")

In [ ]:
sample_text = "Machine Learning is fascinating"
vector = embeddings.embed_query(sample_text)
print(f"Text: {sample_text}")
print(f"Embedding length: {len(vector)}")
print(vector[:10])

In [ ]:
vector

### Initialize the ChromaDB Vector Store

In [ ]:
chunks

In [ ]:
## Create a ChromaDB vector store
persist_directory = tempfile.mkdtemp(prefix="chroma_db_")

## Initialize ChromaDB with the configured embeddings
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=persist_directory,
    collection_name="rag_collection",
)

print(f"Vector store created with {vectorstore._collection.count()} vectors")
print(f"Persisted to: {persist_directory}")

### Test Similarity Search

In [ ]:
query="What are the types of machine learning?"

similar_docs=vectorstore.similarity_search(query,k=3)
similar_docs

In [ ]:
query="what is NLP?"

similar_docs=vectorstore.similarity_search(query,k=3)
similar_docs

In [ ]:
query="what is Deep Learning?"

similar_docs=vectorstore.similarity_search(query,k=3)
similar_docs

In [ ]:
print(f"Query: {query}")
print(f"\nTop {len(similar_docs)} similar chunks:")
for i, doc in enumerate(similar_docs):
    print(f"\n--- Chunk {i+1} ---")
    print(doc.page_content[:200] + "...")
    print(f"Source: {doc.metadata.get('source', 'Unknown')}")

### Similarity Search with Scores

In [ ]:
results_scores=vectorstore.similarity_search_with_score(query,k=3)
results_scores

#### Understanding Similarity Scores
Similarity scores show how close a document chunk is to the query. The interpretation depends on the distance metric.

ChromaDB default: L2 distance (Euclidean distance)

- Lower scores mean closer matches
- A score of 0 means identical vectors
- Typical values are often between 0 and 2, but they can be higher

Cosine similarity, if configured:

- Higher scores mean closer matches
- Values range from -1 to 1, where 1 is the closest match


#### Initialize the LLM, Prompt, and RAG Chain

In [ ]:
from langchain_core.language_models.fake_chat_models import FakeListChatModel

if OPENAI_API_KEY:
    from langchain_openai import ChatOpenAI
    llm = ChatOpenAI(model="gpt-3.5-turbo")
    print("Using ChatOpenAI from the API.")
else:
    llm = FakeListChatModel(
        responses=[
            "Demo answer generated without an external LLM. Add an API key for live model responses."
        ]
    )
    print("OPENAI_API_KEY not found. Using FakeListChatModel so the notebook still runs.")

In [ ]:
test_response = llm.invoke("What is Large Language Models")
test_response

In [ ]:
llm

In [ ]:
llm.invoke("What is AI")

### Modern RAG Chain

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

In [ ]:
## Convert vector store to retriever
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}  # Retrieve top 3 relevant chunks
)
retriever

In [ ]:
## Create a prompt template
from langchain_core.prompts import ChatPromptTemplate
system_prompt="""You are an assistant for question-answering tasks. 
Use the following pieces of retrieved context to answer the question. 
If you don't know the answer, just say that you don't know. 
Use three sentences maximum and keep the answer concise.

Context: {context}"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}")
])

In [ ]:
prompt

##### What is `create_stuff_documents_chain`?
`create_stuff_documents_chain` inserts retrieved documents into a prompt's context field and sends that prompt to the LLM.


In [ ]:
### Create a document chain
def format_docs(docs):
    separator = chr(10) * 2
    return separator.join(doc.page_content for doc in docs)

document_chain = prompt | llm | StrOutputParser()
document_chain

This chain:

- Takes retrieved documents
- Inserts them into the prompt's `{context}` placeholder
- Sends the prompt to the LLM
- Returns the LLM response


#### What is `create_retrieval_chain`?
`create_retrieval_chain` combines a retriever with a document chain so a query can fetch context before the LLM answers.


In [ ]:
### Create The Final RAG Chain
def run_rag(inputs):
    question = inputs["input"]
    docs = retriever.invoke(question)
    answer = document_chain.invoke({"input": question, "context": format_docs(docs)})
    return {"input": question, "context": docs, "answer": answer}

rag_chain = RunnableLambda(run_rag)
rag_chain

In [ ]:
response=rag_chain.invoke({"input":"What is Deep Learning"})

In [ ]:
response

In [ ]:
response['answer']

In [ ]:
# Function to query the modern RAG system
def query_rag_modern(question):
    print(f"Question: {question}")
    print("-" * 50)
    
    # Using create_retrieval_chain approach
    result = rag_chain.invoke({"input": question})
    
    print(f"Answer: {result['answer']}")
    print("\nRetrieved Context:")
    for i, doc in enumerate(result['context']):
        print(f"\n--- Source {i+1} ---")
        print(doc.page_content[:200] + "...")
    
    return result

# Test queries
test_questions = [
    "What are the three types of machine learning?",
    "What is deep learning and how does it relate to neural networks?",
    "What are CNNs best used for?"
]

for question in test_questions:
    result = query_rag_modern(question)
    print("\n" + "="*80 + "\n")

### RAG Chain with LCEL (LangChain Expression Language)

In [ ]:
# Even more flexible approach using LCEL
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel

In [ ]:
# Create a custom prompt
custom_prompt = ChatPromptTemplate.from_template("""Use the following context to answer the question. 
If you don't know the answer based on the context, say you don't know.
Provide specific details from the context to support your answer.

Context:
{context}

Question: {question}

Answer:""")
custom_prompt

In [ ]:
retriever

In [ ]:
## Format the output documents for the prompt
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [ ]:
## Build the chain using LCEL

rag_chain_lcel=(
    { 
        "context":retriever | format_docs,
        "question": RunnablePassthrough()
     }
    | custom_prompt
    | llm
    | StrOutputParser()
)

rag_chain_lcel

In [ ]:
response=rag_chain_lcel.invoke("What is Deep Learning")
response

In [ ]:
retriever.invoke("What is Deep Learning")

In [ ]:
# Query using the LCEL approach
def query_rag_lcel(question):
    print(f"Question: {question}")
    print("-" * 50)

    answer = rag_chain_lcel.invoke(question)
    print(f"Answer: {answer}")

    docs = retriever.invoke(question)
    print()
    print("Source Documents:")
    for i, doc in enumerate(docs):
        print()
        print(f"--- Source {i+1} ---")
        print(doc.page_content[:200] + "...")

In [ ]:
# Test LCEL chain
print("Testing LCEL Chain:")
query_rag_lcel("What are the key concepts in reinforcement learning?")

In [ ]:
query_rag_lcel("What is machine learning?")

In [ ]:
query_rag_lcel("What is deep learning?")

### Add Documents to an Existing Vector Store

In [ ]:
vectorstore

In [ ]:
# Add new documents to the existing vector store
new_document = """
Reinforcement Learning in Detail

Reinforcement learning (RL) is a type of machine learning where an agent learns to make 
decisions by interacting with an environment. The agent receives rewards or penalties 
based on its actions and learns to maximize cumulative reward over time. Key concepts 
in RL include: states, actions, rewards, policies, and value functions. Popular RL 
algorithms include Q-learning, Deep Q-Networks (DQN), Policy Gradient methods, and 
Actor-Critic methods. RL has been successfully applied to game playing (like AlphaGo), 
robotics, and autonomous systems.
"""

In [ ]:
new_document

In [ ]:
chunks

In [ ]:
new_doc=Document(
    page_content=new_document,
    metadata={"source": "manual_addition", "topic": "reinforcement_learning"}
)

In [ ]:
new_doc

In [ ]:
## split the documents
new_chunks=text_splitter.split_documents([new_doc])
new_chunks

In [ ]:
### Add new documents to vectorstore
vectorstore.add_documents(new_chunks)



In [ ]:
print(f"Added {len(new_chunks)} new chunks to the vector store")
print(f"Total vectors now: {vectorstore._collection.count()}")

In [ ]:
## query with the updated vector
new_question="What are the keys concepts in reinforcement learning"
result=query_rag_lcel(new_question)
result

### Conversational RAG Memory
Conversational memory helps RAG systems handle follow-up questions that depend on earlier turns.

Common cases:

- Follow-up questions that reference previous answers
- Pronouns such as "it", "they", and "that"
- Context-dependent queries that build on prior discussion

Traditional RAG retrieves documents from the current query only. A history-aware retriever first rewrites follow-up questions as standalone queries, then retrieves context for the rewritten query.


- `create_history_aware_retriever`: rewrites context-dependent questions before retrieval
- `MessagesPlaceholder`: inserts chat history into prompts
- `HumanMessage` and `AIMessage`: structured message types for conversation history


In [ ]:
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.runnables import RunnableLambda

In [ ]:
def standalone_question(inputs):
    question = inputs["input"]
    history = inputs.get("chat_history", [])
    if history and any(term in question.lower() for term in ["it", "its", "they", "that"]):
        last_human = next(
            (message.content for message in reversed(history) if isinstance(message, HumanMessage)),
            "",
        )
        if last_human:
            return f"Previous question: {last_human}. Follow-up: {question}"
    return question

In [ ]:
## create history aware retriever
history_aware_retriever = RunnableLambda(
    lambda inputs: retriever.invoke(standalone_question(inputs))
)
history_aware_retriever

In [ ]:
# Create a conversational RAG chain with the lightweight history-aware retriever.
def run_conversational_rag(inputs):
    question = inputs["input"]
    docs = history_aware_retriever.invoke(inputs)
    answer = document_chain.invoke({"input": question, "context": format_docs(docs)})
    return {"input": question, "context": docs, "answer": answer}

conversational_rag_chain = RunnableLambda(run_conversational_rag)
print("Conversational RAG chain created!")

In [ ]:
chat_history=[]
# First question
result1 = conversational_rag_chain.invoke({
    "chat_history": chat_history,
    "input": "What is machine learning?"
})
print(f"Q: What is machine learning?")
print(f"A: {result1['answer']}")

In [ ]:
chat_history.extend([
    HumanMessage(content="What is machine learning"),
    AIMessage(content=result1['answer'])
])

In [ ]:
chat_history

In [ ]:
## Follow up question
# Follow-up question
result2 = conversational_rag_chain.invoke({
    "chat_history": chat_history,
    "input": "What are its main types?"  # Refers to ML from previous question
})
result2

In [ ]:
result2['answer']

### Using Groq LLMs
 

In [ ]:
llm

In [ ]:
load_dotenv()

In [ ]:
os.getenv("GROQ_API_KEY")

In [ ]:
from langchain_core.language_models.fake_chat_models import FakeListChatModel
from langchain.chat_models import init_chat_model

In [ ]:
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
if GROQ_API_KEY:
    llm = init_chat_model(model="groq:gemma2-9b-it")
    print("Using Groq model from the API.")
else:
    llm = FakeListChatModel(
        responses=["Demo Groq fallback response. Add GROQ_API_KEY for live responses."]
    )
    print("GROQ_API_KEY not found. Using FakeListChatModel so the notebook still runs.")

In [ ]:
llm

In [ ]:
llm.invoke("Hello from the configured LLM")